# 02a — The McCulloch-Pitts Neuron

This notebook builds the McCulloch-Pitts (MP) neuron from scratch in pure Python.

The MP neuron (1943) is the first mathematical model of a biological neuron.
It has no learning — weights and threshold are set by hand.
Its purpose is to show that logical computation can be done by neuron-like units.

**What we build:**
- The MP neuron model
- AND, OR, NOT gates
- The critical limitation: it cannot learn

---
## Part 1: The MP Neuron Model

```
x₁ ──(w₁)──┐
            ├──► z = Σ wᵢxᵢ ──► threshold ──► output ∈ {0, 1}
x₂ ──(w₂)──┘
```

- Inputs: binary (0 or 1)
- Weights: fixed integers (set by designer)
- Threshold θ: fixed (set by designer)
- Output: 1 if z ≥ θ, else 0

In [3]:
def mp_neuron(inputs, weights, threshold):
    """
    McCulloch-Pitts neuron.

    inputs    : list of binary values (0 or 1)
    weights   : list of integers (fixed, set by designer)
    threshold : integer (fixed, set by designer)

    Returns 1 if weighted sum >= threshold, else 0.
    """
    z = sum(w * x for w, x in zip(weights, inputs))
    return 1 if z >= threshold else 0

print('MP Neuron defined.')
print('Inputs: binary only (0 or 1)')
print('Weights: fixed by designer')
print('Threshold: fixed by designer')
print('Output: 1 if weighted_sum >= threshold, else 0')

MP Neuron defined.
Inputs: binary only (0 or 1)
Weights: fixed by designer
Threshold: fixed by designer
Output: 1 if weighted_sum >= threshold, else 0


---
## Part 2: AND Gate

AND is true only when **both** inputs are 1.

```
x₁  x₂  AND
 0   0    0
 0   1    0
 1   0    0
 1   1    1
```

Setting: w₁ = 1, w₂ = 1, θ = 2
Both inputs must fire (sum = 2) to reach threshold.

In [4]:
# AND gate: w1=1, w2=1, threshold=2
weights_and   = [1, 1]
threshold_and = 2

inputs_all = [(0,0), (0,1), (1,0), (1,1)]

print('AND Gate  (w1=1, w2=1, threshold=2)')
print('-' * 40)
print(f'  {"x1":>4}  {"x2":>4}  {"z":>4}  {"output":>8}  {"correct?":>9}')

true_and = [0, 0, 0, 1]

for (x1, x2), true_out in zip(inputs_all, true_and):
    z      = weights_and[0]*x1 + weights_and[1]*x2
    output = mp_neuron([x1, x2], weights_and, threshold_and)
    check  = '✓' if output == true_out else '✗'
    print(f'  {x1:>4}  {x2:>4}  {z:>4}  {output:>8}  {check:>9}')

AND Gate  (w1=1, w2=1, threshold=2)
----------------------------------------
    x1    x2     z    output   correct?
     0     0     0         0          ✓
     0     1     1         0          ✓
     1     0     1         0          ✓
     1     1     2         1          ✓


---
## Part 3: OR Gate

OR is true when **at least one** input is 1.

Setting: w₁ = 1, w₂ = 1, θ = 1
Either input firing (sum ≥ 1) is enough.

In [5]:
# OR gate: w1=1, w2=1, threshold=1
weights_or   = [1, 1]
threshold_or = 1

print('OR Gate  (w1=1, w2=1, threshold=1)')
print('-' * 40)
print(f'  {"x1":>4}  {"x2":>4}  {"z":>4}  {"output":>8}  {"correct?":>9}')

true_or = [0, 1, 1, 1]

for (x1, x2), true_out in zip(inputs_all, true_or):
    z      = weights_or[0]*x1 + weights_or[1]*x2
    output = mp_neuron([x1, x2], weights_or, threshold_or)
    check  = '✓' if output == true_out else '✗'
    print(f'  {x1:>4}  {x2:>4}  {z:>4}  {output:>8}  {check:>9}')

OR Gate  (w1=1, w2=1, threshold=1)
----------------------------------------
    x1    x2     z    output   correct?
     0     0     0         0          ✓
     0     1     1         1          ✓
     1     0     1         1          ✓
     1     1     2         1          ✓


---
## Part 4: NOT Gate

NOT inverts a single input.

Setting: w₁ = -1, θ = 0
A negative (inhibitory) weight — when input fires, sum goes negative, below threshold.

In [6]:
# NOT gate: w1=-1, threshold=0
weights_not   = [-1]
threshold_not = 0

print('NOT Gate  (w1=-1, threshold=0)')
print('-' * 35)
print(f'  {"x1":>4}  {"z":>4}  {"output":>8}  {"correct?":>9}')

true_not = [1, 0]

for x1, true_out in zip([0, 1], true_not):
    z      = weights_not[0] * x1
    output = mp_neuron([x1], weights_not, threshold_not)
    check  = '✓' if output == true_out else '✗'
    print(f'  {x1:>4}  {z:>4}  {output:>8}  {check:>9}')

print()
print('Negative weights act as inhibitory connections.')
print('When x1=1: z = -1 < 0 → neuron stays silent → output = 0 (NOT 1 = 0)')
print('When x1=0: z =  0 ≥ 0 → neuron fires        → output = 1 (NOT 0 = 1)')

NOT Gate  (w1=-1, threshold=0)
-----------------------------------
    x1     z    output   correct?
     0     0         1          ✓
     1    -1         0          ✓

Negative weights act as inhibitory connections.
When x1=1: z = -1 < 0 → neuron stays silent → output = 0 (NOT 1 = 0)
When x1=0: z =  0 ≥ 0 → neuron fires        → output = 1 (NOT 0 = 1)


---
## Part 5: Building NAND from AND + NOT

NAND = NOT AND. We can compose MP neurons to build more complex logic.
This shows that networks of MP neurons are computationally universal.

In [7]:
# NAND = NOT(AND(x1, x2))
# Two-stage computation: AND first, then NOT

def nand_gate(x1, x2):
    and_output = mp_neuron([x1, x2], weights=[1, 1], threshold=2)
    nand_output = mp_neuron([and_output], weights=[-1], threshold=0)
    return nand_output

print('NAND Gate  (AND → NOT, two MP neurons in sequence)')
print('-' * 45)
print(f'  {"x1":>4}  {"x2":>4}  {"AND":>6}  {"NAND":>6}  {"correct?":>9}')

true_nand = [1, 1, 1, 0]

for (x1, x2), true_out in zip(inputs_all, true_nand):
    and_out  = mp_neuron([x1, x2], [1, 1], 2)
    nand_out = nand_gate(x1, x2)
    check    = '✓' if nand_out == true_out else '✗'
    print(f'  {x1:>4}  {x2:>4}  {and_out:>6}  {nand_out:>6}  {check:>9}')

print()
print('NAND is functionally complete — any logic circuit can be built from NAND gates.')
print('Therefore: networks of MP neurons can compute any logical function.')

NAND Gate  (AND → NOT, two MP neurons in sequence)
---------------------------------------------
    x1    x2     AND    NAND   correct?
     0     0       0       1          ✓
     0     1       0       1          ✓
     1     0       0       1          ✓
     1     1       1       0          ✓

NAND is functionally complete — any logic circuit can be built from NAND gates.
Therefore: networks of MP neurons can compute any logical function.


---
## Part 6: The Critical Limitation

The MP neuron cannot learn. Weights and threshold are set by hand.
What happens when we do not know the right weights?

In [8]:
# Try to find weights for a new problem by brute force
# Task: output 1 if x1 + x2 + x3 >= 2 (majority vote)

majority_data = [
    ([0,0,0], 0),
    ([0,0,1], 0),
    ([0,1,0], 0),
    ([1,0,0], 0),
    ([0,1,1], 1),
    ([1,0,1], 1),
    ([1,1,0], 1),
    ([1,1,1], 1),
]

print('Task: majority vote (output 1 if at least 2 of 3 inputs are 1)')
print()
print('With MP neuron, we must search for the right weights manually.')
print('Trying w1=1, w2=1, w3=1, threshold=2 ...')
print()

w = [1, 1, 1]
theta = 2

print(f'  {"inputs":>12}  {"output":>8}  {"correct?":>9}')
all_correct = True
for inputs, true_out in majority_data:
    output = mp_neuron(inputs, w, theta)
    check  = '✓' if output == true_out else '✗'
    if output != true_out:
        all_correct = False
    print(f'  {str(inputs):>12}  {output:>8}  {check:>9}')

print()
if all_correct:
    print('Correct — but we had to know the answer to set the weights.')
print('The MP neuron has no mechanism to find these weights from data.')
print('That is the gap the Perceptron fills — see notebook 02b.')

Task: majority vote (output 1 if at least 2 of 3 inputs are 1)

With MP neuron, we must search for the right weights manually.
Trying w1=1, w2=1, w3=1, threshold=2 ...

        inputs    output   correct?
     [0, 0, 0]         0          ✓
     [0, 0, 1]         0          ✓
     [0, 1, 0]         0          ✓
     [1, 0, 0]         0          ✓
     [0, 1, 1]         1          ✓
     [1, 0, 1]         1          ✓
     [1, 1, 0]         1          ✓
     [1, 1, 1]         1          ✓

Correct — but we had to know the answer to set the weights.
The MP neuron has no mechanism to find these weights from data.
That is the gap the Perceptron fills — see notebook 02b.


---
## Summary

| Property | McCulloch-Pitts Neuron |
|---|---|
| Inputs | Binary (0 or 1) |
| Weights | Fixed integers — set by designer |
| Threshold | Fixed — set by designer |
| Learning | None |
| Can compute | Any logical function (with enough neurons) |
| Cannot do | Learn from data |

The MP neuron proved that neuron-like units can compute logic. 
The Perceptron (1958) extended this by making weights **learnable from data**.